In [44]:
# import pandas as pd
# import numpy as np
# import os
# import  glob

# !pip install duckdb

# note tables so far
# stocks
# stocks_clean
# stocks_features
# stocks_features_clean
# stocks_futures_labeled

# **PHASE 1 — DATA FOUNDATION**
- Let get all trade summary file from local folder
- merge them into single master csv data file
- normalize column names for better understanding and model standards

Company Name| Symbol| Share Volume| Trade Volume| Previous Close (Rs.)| Open (Rs.)| High (Rs.)|Low (Rs.)| **Last Trade (Rs.)| Change (Rs.)| Change (%)

##### Before modeling, you should now write a schema mapper that converts this raw exchange format → standardized ML format
Last Trade	    >  close<br>
Previous Close	>  prev_close<br>
Share Volume	>  volume<br>
Trade Volume	>  trades<br>

### Scan "C:\Users\Lapmart\Downloads\CSE" folder and crate master data set

## Data handling with pandas
- slower when having millions of data

In [45]:
import pandas as pd
import os
import glob
from datetime import datetime

# To see all columns, if not then we see '...' in middle
pd.set_option('display.max_columns', None)

# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE"
OUTPUT = "MASTER_CSE_DATA.csv"

os.makedirs("log", exist_ok=True)
LOG_FILE = LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")


# ================ HELPERS =============

def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        log(f"### Error exctracting name from {filename}")
        return None

# Column mapping
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}


# ================ LOAD FILES ========================

files = glob.glob(os.path.join(FOLDER, "*.csv"))

log("=== LOADER START USING PANDAS FOR DATA ===")
log(f"Total files found: {len(files)}")

valid_frames = []
valid_dates = []

for file in files:
    name = os.path.basename(file)

     # ---- check filename date
    date = extract_date(name)
    if date is None:
        log(f"INVALID DATE FORMAT → skipped: {name}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
    except Exception as e:
        log(f"### READ ERROR → {name} → {e}")
        continue
        
    # ---- empty file check
    if df.empty:
        log(f"EMPTY FILE → skipped: {name}")
        continue

    # ---- normalize column names
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    # ---- add date column
    df["date"] = date

    valid_frames.append(df)
    valid_dates.append(date)


# ================ MERGE =======================

if not valid_frames:
    log("NO VALID FILES FOUND")
    exit()
    
# combine all files
master = pd.concat(valid_frames, ignore_index=True)


# ================ DUPLICATE CHECK =====================

dupes = master.duplicated(subset=["symbol","date"])
dup_count = dupes.sum()

if dup_count > 0:
    log(f"DUPLICATES FOUND: {dup_count} rows removed")
    master = master[~dupes]


# ================ SORT ========================

master = master.sort_values(["symbol", "date"])


# # ================ MISSING DATE REPORT ==================

# valid_dates = sorted(set(valid_dates))
# missing_days = []

# for i in range(len(valid_dates)-1):
#     diff = (valid_dates[i+1] - valid_dates[i]).days
#     if diff > 1:
#         missing_days.append((valid_dates[i], valid_dates[i+1]))

# if missing_days:
#     log("MISSING DATE GAPS:")
#     for g in missing_days:
#         log(f"{g[0].date()} → {g[1].date()}")


# ================ SAVE =======================
        
master.to_csv(OUTPUT, index=False)

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED ===")
print("DONE — Master dataset created using pandas")


=== LOADER START USING PANDAS FOR DATA ===
Total files found: 54
DUPLICATES FOUND: 293 rows removed
MASTER DATASET CREATED → MASTER_CSE_DATA.csv
=== LOADER FINISHED ===
DONE — Master dataset created using pandas


## Data Hndling with DuckDB

### Why Data Engineers Prefer DuckDB
- Real data pipelines often use it for:
- stock market analysis
- log processing
- machine learning datasets
- financial backtesting
- Because it handles millions to billions of rows locally.

In [46]:
import pandas as pd
import os
import glob # finds files/folders matching patterns like wildcards in Python
from datetime import datetime
import duckdb



# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE" # my local stored folder path
OUTPUT = "MASTER_CSE_DATA.csv" # Name of the master data file

os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")



# ================ HELPERS =================

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

# Method to extract date from file name in format like 260210_tradesummary.csv.
def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        return None

# Standardize column names, helper map for mapping existing unusual column names into standard names 
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}



# ================= CONNECT DUCKDB =================

con = duckdb.connect(database="cse_data.db")  # Create/connect DB
log("Creating DuckDB table 'stocks' ...")

con.execute("""
CREATE TABLE IF NOT EXISTS stocks (
    company TEXT,
    symbol TEXT,
    low DOUBLE,
    close DOUBLE,
    prev_close DOUBLE,
    open DOUBLE,
    high DOUBLE,
    volume DOUBLE,
    trades DOUBLE,
    change DOUBLE,
    change_percentage DOUBLE,
    date DATE
)
""")



# ================= LOAD CSVs DIRECTLY =================

log("=== LOADER START WITH DUCKDB FOR DATA HANDLING ===")

files = glob.glob(os.path.join(FOLDER, "*.csv"))
log(f"Total files found: {len(files)}")

for file in files:
    filename = os.path.basename(file) # extract base name
    date = extract_date(filename) # get date from filename though function extract_date()
    
    if date is None:
        log(f"### INVALID DATE FORMAT → skipped: {filename}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
        if df.empty:
            log(f"EMPTY FILE → skipped: {filename}")
            continue
    except Exception as e:
        log(f"READ ERROR → {filename} → {e}")
        continue
        
    # ---- Standardize column names, We assign a new list to df.columns to update the column names after processing them
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True) # Renames columns based on COLUMN_MAP.


    # ======== DATA QUALITY REPORT =========
    
    no_of_total_rows = len(df)
    no_of_missing_symbol = df["symbol"].isna().sum() if "symbol" in df.columns else total_rows
    no_of_missing_close = df["close"].isna().sum() if "close" in df.columns else total_rows

    numeric_cols = ["low","close","prev_close","open","high","volume","trades","change","change_percentage"]
    invalid_numeric = {}

    for col in numeric_cols:
        if col in df.columns:
            invalid_numeric[col] = pd.to_numeric(df[col], errors="coerce").isna().sum()
            
    if(no_of_missing_symbol > 0 or no_of_missing_close > 0):
        log("\n============== DATA QUALITY REPORT ==============")
        log(f"FILE: {filename}")
        log(f"Total rows: {no_of_total_rows}")
        log(f"Missing symbols: {no_of_missing_symbol}")
        log(f"Missing close prices: {no_of_missing_close}")
        log("============== ENF OF DATA QUALITY REPORT ==============\n")

    for col, count in invalid_numeric.items():
        if count > 0:
            log(f"Invalid numeric values in {col}: {count}")

    # check negative prices
    if "close" in df.columns:
        neg_price = (pd.to_numeric(df["close"], errors="coerce") <= 0).sum()
        if neg_price > 0:
            log(f"Invalid negative prices: {neg_price}")

    
    # ================= CLEAN DATA =================

    no_of_rows_before_clean = len(df)

    # remove rows where symbol missing BEFORE casting to string
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        
    # strip spaces
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        df["symbol"] = df["symbol"].astype(str).str.strip()

    if "company" in df.columns:
        df["company"] = df["company"].astype(str).str.strip()
    
    # convert numeric columns
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # remove rows where close price missing
    if "close" in df.columns:
        df = df[df["close"].notna()]

    # remove impossible values
    if "close" in df.columns:
        df = df[df["close"] > 0]

    if "volume" in df.columns:
        df = df[df["volume"] >= 0]

    # fix percentage anomalies
    if "change_percentage" in df.columns:
        df.loc[(df["change_percentage"] < -100) | (df["change_percentage"] > 1000), "change_percentage"] = None

    # drop fully empty rows
    df = df.dropna(how="all")

    no_of_rows_after_clean = len(df)
    
    if(no_of_rows_before_clean > no_of_rows_after_clean):
        log("\n========== DATA CLEAN REPORT ========")
        log(f"Some rows removed, File : {filename} before cleaning → {no_of_rows_before_clean} rows, After {no_of_rows_after_clean} rows\n")

    if len(df) == 0:
        log(f"ALL ROWS REMOVED AFTER CLEANING → skipped {filename}")
        continue
    
    # ---- add date column
    df["date"] = date

    # ------- Insert directly into DuckDB 'stocks' table
    con.register("temp_df", df)
    con.execute("INSERT INTO stocks SELECT * FROM temp_df")
    # log(f"Inserted {len(df)} rows from {filename}")

log("All CSVs inserted into DuckDB.")



# ================  REMOVE DUPLICATES =====================
# Removes duplicate rows for the same symbol and date
# numbers duplicate rows starting from 1
# WHERE rn = 1 → keeps only the first row of each symbol/date combination

log("Removing duplicates...")

# Creates a table named stocks_clean
# If it already exists → deletes old one and recreates it
# add ne column rn, assign occurnce counnt for each record only keeep occurence number 1 others will be delted
# 
con.execute("""
CREATE OR REPLACE TABLE stocks_clean AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY symbol, date
               ORDER BY date
           ) AS rn
    FROM stocks
)
WHERE rn = 1
""")



# ================ SORT ========================

log("Sorting dataset...")

# Sorts the cleaned dataset by symbol then date
con.execute("""
CREATE OR REPLACE TABLE stocks_sorted AS
SELECT *
FROM stocks_clean
ORDER BY symbol, date
""")



# ================ CREATE INDEX ========================

log("Creating composite index on symbol+date for speed...")

# Adds a composite index on (symbol, date)
# Speeds up all queries that filter by symbol, date, or both
con.execute("""
CREATE INDEX IF NOT EXISTS idx_symbol_date ON stocks_sorted(symbol, date)
""")



# ================ MISSING DATE REPORT ==================
# TODO : if needed



# ================ EXPORT =======================

log("Exporting final CSV...")

con.execute(f"""
COPY stocks_sorted TO '{OUTPUT}' (HEADER, DELIMITER ',')
""")

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED USING DUCKDB ===")
print("DONE — Master dataset created")

Creating DuckDB table 'stocks' ...
=== LOADER START WITH DUCKDB FOR DATA HANDLING ===
Total files found: 54

============== DATA QUALITY REPORT ==============
FILE: 260228_trade-summary-test-dup.csv
Total rows: 293
Missing symbols: 1
Missing close prices: 1
============== ENF OF DATA QUALITY REPORT ==============

Invalid numeric values in close: 1

========== DATA CLEAN REPORT ========
Some rows removed, File : 260228_trade-summary-test-dup.csv before cleaning → 293 rows, After 291 rows


============== DATA QUALITY REPORT ==============
FILE: 260228_trade-summary-test.csv
Total rows: 293
Missing symbols: 1
Missing close prices: 1
============== ENF OF DATA QUALITY REPORT ==============

Invalid numeric values in close: 1

========== DATA CLEAN REPORT ========
Some rows removed, File : 260228_trade-summary-test.csv before cleaning → 293 rows, After 291 rows

All CSVs inserted into DuckDB.
Removing duplicates...
Sorting dataset...
Creating composite index on symbol+date for speed...
Ex

### Test speed of query execution with indexing

In [47]:
# commnet above code's indexing part and give a dofferent name for db cvvration try it with below code
# try agian with uncommenting

import time
import duckdb

# Change this to the DB you want to test
con = duckdb.connect("cse_data.db")  # or 'cse_data_index.db'

query = "SELECT * FROM stocks_sorted WHERE symbol='JKH'"

start = time.time()
results = con.execute(query).fetchall()
end = time.time()

print(f"Rows returned: {len(results)}")
print(f"Query time: {end - start:.4f} seconds")

Rows returned: 0
Query time: 0.0018 seconds


In [48]:
df.columns

Index(['company', 'symbol', 'volume', 'trades', 'prev_close', 'open', 'high',
       'low', 'close', 'change', 'change_percentage', 'date'],
      dtype='object')

In [49]:
df = pd.read_csv(r"C:\Users\Lapmart\Jupyter AI Models\CSE Investment Assistence/MASTER_CSE_DATA.csv")
df.head()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
0,ASIA ASSET FINANCE PLC,AAF.N0000,184992.0,102.0,61.7,61.5,63.0,60.8,60.9,-0.8,-1.30,2025-10-23,1
1,ASIA ASSET FINANCE PLC,AAF.N0000,225496.0,128.0,60.9,61.6,62.0,60.0,60.3,-0.6,-0.99,2025-10-24,1
2,ASIA ASSET FINANCE PLC,AAF.N0000,217100.0,171.0,60.3,61.0,61.0,58.5,59.2,-1.1,-1.82,2025-10-27,1
3,ASIA ASSET FINANCE PLC,AAF.N0000,107995.0,93.0,59.2,60.9,61.0,59.0,59.9,0.7,1.18,2025-10-28,1
4,ASIA ASSET FINANCE PLC,AAF.N0000,217448.0,101.0,59.9,60.0,62.0,59.0,60.5,0.6,1.00,2025-10-29,1


In [50]:
df.tail()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
15124,YORK ARCADE HOLDINGS PLC,YORK.N0000,131286.0,143.0,16.8,16.8,17.2,16.5,16.6,-0.2,-1.19,2026-02-24,1
15125,YORK ARCADE HOLDINGS PLC,YORK.N0000,240721.0,251.0,16.6,16.9,16.9,16.2,16.3,-0.3,-1.81,2026-02-25,1
15126,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-26,1
15127,YORK ARCADE HOLDINGS PLC,YORK.N0000,414474.0,215.0,16.1,16.3,17.2,16.0,16.3,0.2,1.24,2026-02-27,1
15128,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-28,1


# **PHASE 2 — FEATURE ENGINEERING ENGINE**
- For 10 day profit
- let create new table **stocks_features** > reads data from table **stocks_sorted** > copies all original columns > Adds new calculated columns
- WINDOW w → defines a reusable window that we can use any where insted of typing long query several times,it say defines how rows are grouped and ordered
  - grouped by stock symbol
  - ordered by date

In [51]:
# ============== Create Feature Table Base ===========================================================

log("Creating feature base table...")

# Opens a connection to a DuckDB database file named cse_data.db, If it doesn’t exist, DuckDB creates it
con = duckdb.connect(database="cse_data.db")

# Creates a table called stocks_features, If it already exists → replaces it
# SELECT *,  > Selects all columns
# LAG window function looks backward in the result set, returns the value of the close column from 5 rows earlier

# close_5d     --> price 5 day ealier 
# close_10d    --> price 10 ddays earlier, 
# high_20      --> highest closing price from the last 20 rows (including today)
# low_20       --> lowest closing price from the last 20 rows (including today)
# avg_vol_20   --> average volume over the last 20 records including today
# std_vol_20   --> standard deviation, how stable or unstable volume has been recently(how much the trading volume has been changing(varying) over the 
#                  last 20 rows including today) if small : volume is stable if large : volume is fluctuating a lot 
#                  how to cal : Find the average of last 20 rec > Find difference from average of each > Square differences(ignore - vals) 
#                  > Average of squared values > Square root 
# std_close_10 --> how much the closing price has been changing (varying) over the last 10 rows including today
con.execute("""
CREATE OR REPLACE TABLE stocks_features AS
SELECT
    *,
    
    LAG(close, 5)  OVER w AS close_5d,
    LAG(close, 10) OVER w AS close_10d,

    MAX(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS high_20,
    MIN(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS low_20,

    AVG(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS avg_vol_20,
    STDDEV(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS std_vol_20,

    STDDEV(close) OVER (w ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) AS std_close_10

FROM stocks_sorted
WINDOW w AS (PARTITION BY symbol ORDER BY date);
""")


# =================== Creating Momentum Features ==========================================================

# Momentum = how fast price is changing
# ret_5  --> Calculates 5-day return as (Current price ÷ price 5 days ago) − 1 >>> Without -1 : 120 / 100 = 1.2, but we need growth  as 0.2, the use -1
# ret_10 --> Calculate 10-day return
log("Creating momentum features...")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_5 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_5 = (close / close_5d) - 1;
""")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_10 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_10 = (close / close_10d) - 1;
""")


# =================== Creating Liquidity Features =========================================================

# how easy it is to buy/sell stock
# volume_ratio --> volume_ratio = volume / avg_vol_20 >>> If >1 → today volume is higher than normal, If <1 → lower than normal 
# volume_z     --> how unusual today’s volume is, measured in units of normal variation
#                  (volume - avg_vol_20) / std_vol_20, 
#                  volume - avg_vol_20 >>> How much today’s volume differs from the recent average.
#                  / std_vol_20 >>> normalizes the difference in terms of “typical variation.”
#                  > 0 → volume higher than normal, < 0 → volume lower than normal, ≈ 0 → volume is typical
log("Creating liquidity features...")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_ratio DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_ratio = volume / avg_vol_20;
""")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_z DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_z = (volume - avg_vol_20) / (std_vol_20 + 1e-9);
""")


# =================== Creating Breakout Context =========================================================

# range_position >>> where today’s price lies in 20-day range
#                    0 = at 20-day low, 0.5 = middle, 1 = at 20-day high
# detect => breakouts, overbought zones, support/resistance
con.execute("ALTER TABLE stocks_features ADD COLUMN range_position DOUBLE;")

con.execute("""
UPDATE stocks_features
SET range_position = (close - low_20) / (high_20 - low_20);
""")


# =================== Remove Invalid Rows =======================================================================

"""We create a new table so the original data stays safe and unchanged for debugging or reuse.
It also makes the ML pipeline cleaner and reproducible by separating raw data from cleaned data"""
# AT THIS POINT WE CREATE NEW TABLE CALLED "stocks_features_clean"
# Creates a cleaned version of the table
# Removes rows where:
#            close_10d IS NULL
#            avg_vol_20 IS NULL
#            std_vol_20 IS NULL
#            std_close_10 IS NULL
#            volume > 0
#            close > 0
# Why cleaning need ? Because early rows don't have enough past data to calculate rolling features
log("Cleaning feature table...")

con.execute("""
CREATE OR REPLACE TABLE stocks_features_clean AS
SELECT *
FROM stocks_features
WHERE
    close_10d IS NOT NULL
    AND avg_vol_20 IS NOT NULL
    AND std_vol_20 IS NOT NULL
    AND std_close_10 IS NOT NULL
    AND volume > 0
    AND close > 0;
""")


# =================== Liquidity Ranking ==============================================================================

# PARTITION / group stocks by day then order by avg_vol_20 in descending order, Look on a particular date, decide who is #1, #2, etc., based on volume
# Highest volume → rank 1, Equal volumes → same rank, Next rank skips number (because of ties,  > 1, 2, 2, 4),  assign values for each relvent row in the 
# liquidity_rankcolumn in table(stocks_features_clean) by matching symbol with its calculated rank, this doesnt change table row order 
# only theliquidity_rank column, this column is not perfect as 1, 2, 3, may be 3, 1, 2
con.execute("ALTER TABLE stocks_features_clean ADD COLUMN liquidity_rank DOUBLE;")

con.execute("""
UPDATE stocks_features_clean t
SET liquidity_rank = r.rank
FROM (
    SELECT
        symbol,
        date,
        RANK() OVER (PARTITION BY date ORDER BY avg_vol_20 DESC) AS rank
    FROM stocks_features_clean
) r
WHERE t.symbol = r.symbol
AND t.date = r.date;
""")


# =================== Export Feature Dataset ============================================================================
# This is the final csv data file used for ML training
log("Exporting feature dataset...")

con.execute("""
COPY stocks_features_clean
TO 'features_master.csv'
(HEADER, DELIMITER ',');
""")

Creating feature base table...
Creating momentum features...
Creating liquidity features...
Cleaning feature table...
Exporting feature dataset...


### Chekc the some info of tempory table "stocks_features"

In [52]:
# Counts all rows in the table
print("all rows : ", con.execute("SELECT COUNT(*) FROM stocks_features").fetchall())

# Counts how many rows are missing the ret_5 value
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features WHERE ret_5 IS NULL").fetchall())

# Counts how many rows are missing the ret_10 value
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features WHERE ret_10 IS NULL").fetchall())

# Finds earliest and latest date in the table
print("data from-to ", con.execute("SELECT MIN(date), MAX(date) FROM stocks_features").fetchall())

all rows :  [(15129,)]
missing for ret_10 :  [(1506,)]
missing for ret_10 :  [(2992,)]
data from-to  [(datetime.date(2025, 10, 23), datetime.date(2026, 2, 28))]


### Check the some info of table "stocks_features_clean"

In [53]:
# Counts all rows in the table
print("all rows : ", con.execute("SELECT COUNT(*) FROM stocks_features").fetchall())

# Counts how many rows are missing the ret_10 value.
print("missing for ret_5 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean WHERE ret_5 IS NULL").fetchall())

# Counts how many rows are missing the ret_10 value.
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean WHERE ret_10 IS NULL").fetchall())

# Finds earliest and latest date in the table.
print("data from-to ", con.execute("SELECT MIN(date), MAX(date) FROM stocks_features_clean").fetchall())

all rows :  [(15129,)]
missing for ret_5 :  [(0,)]
missing for ret_10 :  [(0,)]
data from-to  [(datetime.date(2025, 11, 7), datetime.date(2026, 2, 28))]


### Check table content

In [54]:
import pandas as pd

df = con.execute("""
SELECT *
FROM stocks_features
LIMIT 6;
""").fetchdf()

print(df)

             company     symbol       low   close  prev_close    open   high  \
0  BAIRAHA FARMS PLC  BFL.N0000  249279.0  1254.0      384.75  409.75  470.0   
1  BAIRAHA FARMS PLC  BFL.N0000  137571.0   543.0      458.25  460.00  470.0   
2  BAIRAHA FARMS PLC  BFL.N0000   30153.0   216.0      448.75  450.00  455.0   
3  BAIRAHA FARMS PLC  BFL.N0000   87342.0   317.0      448.25  448.00  448.0   
4  BAIRAHA FARMS PLC  BFL.N0000  113365.0   165.0      442.75  445.00  450.0   
5  BAIRAHA FARMS PLC  BFL.N0000   17267.0   104.0      448.25  450.00  460.0   

   volume  trades  change  change_percentage       date  rn  close_5d  \
0  409.75  458.25   73.50              19.10 2025-10-23   1       NaN   
1  444.00  448.75   -9.50              -2.07 2025-10-24   1       NaN   
2  446.00  448.25   -0.50              -0.11 2025-10-27   1       NaN   
3  440.00  442.75   -5.50              -1.23 2025-10-28   1       NaN   
4  440.00  448.25    5.50               1.24 2025-10-29   1       NaN   
5

In [55]:
df = con.execute("""
SELECT *
FROM stocks_features_clean
LIMIT 5;
""").fetchdf()

print(df)

             company     symbol      low  close  prev_close   open   high  \
0  BAIRAHA FARMS PLC  BFL.N0000  43856.0   82.0      442.25  442.0  452.0   
1  BAIRAHA FARMS PLC  BFL.N0000  18122.0   82.0      449.00  450.0  454.5   
2  BAIRAHA FARMS PLC  BFL.N0000   5998.0   60.0      442.00  442.5  450.0   
3  BAIRAHA FARMS PLC  BFL.N0000  14374.0   55.0      439.75  443.0  449.0   
4  BAIRAHA FARMS PLC  BFL.N0000  32076.0   98.0      445.00  445.0  450.0   

   volume  trades  change  change_percentage       date  rn  close_5d  \
0  440.25  449.00    6.75               1.53 2025-11-07   1     104.0   
1  440.25  449.75    0.75               0.17 2025-11-10   1      95.0   
2  439.00  439.75   -2.25              -0.51 2025-11-12   1      62.0   
3  439.00  445.00    5.25               1.19 2025-11-13   1      43.0   
4  440.00  450.00    5.00               1.12 2025-11-14   1     111.0   

   close_10d  high_20  low_20  avg_vol_20  std_vol_20  std_close_10     ret_5  \
0     1254.0   12

# **PHASE 3 — LABEL CREATION**
- Tranforming historical data into a supervised learning dataset for predicting profitable trades.

In [84]:
# ================ Creating labels for 10-day return ==========================================================
log("=== Phase 3: Creating labels for 10-day return ===")

# Threshold for profitable trade, 0.05 - 5% profit in 10 days
PROFIT_THRESHOLD = 0.05

# Create labeled table with:
#  - future closing price (10 days ahead)
#  - future return %
#  - classification label (buy or not)
#
# f""" allows Python to insert variables like {PROFIT_THRESHOLD} into SQL
#
# Using a CTE (temp) to compute future price once for efficiency
# instead of recalculating LEAD() multiple times
#
# LEAD(close, 10)
#     → gets closing price 10 rows ahead
# PARTITION BY symbol
#     → calculation done separately for each stock
# ORDER BY date
#     → ensures rows are in time order
#
# Return formula:
#     future_return = (future_price / current_price) - 1
#
# Target label rule:
#     return ≥ threshold → 1 (profitable → buy signal)
#     return < threshold → 0 (not profitable)
con.execute(f"""
CREATE OR REPLACE TABLE stocks_features_labeled AS
WITH temp  AS (
    SELECT *,
           LEAD(close, 10) OVER (PARTITION BY symbol ORDER BY date) AS future_close_10d
    FROM stocks_features_clean
    WHERE close > 0
)
SELECT *,
       (future_close_10d / close - 1) AS future_return_10d,
       CASE 
           WHEN (future_close_10d / close - 1) >= {PROFIT_THRESHOLD} THEN 1
           ELSE 0
       END AS target_buy_10
FROM temp;
""")

log("Label creation complete: 10-day future return and target_buy_10 added.")


# ================= EXPORT LABELED DATASET ==================
OUTPUT_LABELS = "features_labeled_10d.csv"
log(f"Exporting labeled dataset to {OUTPUT_LABELS} ...")

# HEADER → includes column names
# DELIMITER ',' → comma-separated format
con.execute(f"""
COPY stocks_features_labeled
TO '{OUTPUT_LABELS}'
(HEADER, DELIMITER ',');
""")

log(f"Labeled dataset exported dataset ready for ML training → {OUTPUT_LABELS}")

df = con.execute("""
SELECT *
FROM stocks_features_labeled
LIMIT 5;
""").fetchdf()

print(df)

=== Phase 3: Creating labels for 10-day return ===
Label creation complete: 10-day future return and target_buy_10 added.
Exporting labeled dataset to features_labeled_10d.csv ...
Labeled dataset exported dataset ready for ML training → features_labeled_10d.csv
                     company      symbol        low   close  prev_close  \
0  BOGALA GRAPHITE LANKA PLC  BOGA.N0000  2204687.0  2919.0      109.75   
1  BOGALA GRAPHITE LANKA PLC  BOGA.N0000  1168707.0  1532.0      137.00   
2  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   771411.0  1772.0      150.00   
3  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   264814.0   654.0      150.50   
4  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   107015.0   356.0      150.75   

    open    high  volume  trades  change  change_percentage       date  rn  \
0  117.0  137.00  115.75  137.00   27.25              24.83 2025-11-07   1   
1  160.0  171.25  152.00  171.25   34.25              25.00 2025-11-10   1   
2  150.0  157.00  137.00  150.50    0.50             

# **Phase 4 — Train Prediction Model**
- Load dataset
- Select features
- Split data
- Train model
- Evaluate performance
- Save mdodel

### Load Dataset

In [85]:
import pandas as pd
# pd.set_option("display.max_columns", None)

df = pd.read_csv("features_labeled_10d.csv")
print(df.shape)
print(df.head())
print(df.columns)

(12137, 29)
                     company      symbol        low   close  prev_close  \
0  BOGALA GRAPHITE LANKA PLC  BOGA.N0000  2204687.0  2919.0      109.75   
1  BOGALA GRAPHITE LANKA PLC  BOGA.N0000  1168707.0  1532.0      137.00   
2  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   771411.0  1772.0      150.00   
3  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   264814.0   654.0      150.50   
4  BOGALA GRAPHITE LANKA PLC  BOGA.N0000   107015.0   356.0      150.75   

    open    high  volume  trades  change  change_percentage        date  rn  \
0  117.0  137.00  115.75  137.00   27.25              24.83  2025-11-07   1   
1  160.0  171.25  152.00  171.25   34.25              25.00  2025-11-10   1   
2  150.0  157.00  137.00  150.50    0.50               0.33  2025-11-12   1   
3  150.0  157.00  141.00  150.75    0.25               0.17  2025-11-13   1   
4  150.0  154.00  146.25  148.50   -2.25              -1.49  2025-11-14   1   

   close_5d  close_10d  high_20  low_20  avg_vol_20  std_vol_2

### Select Features + Target

In [86]:
features = [
    "ret_5",
    "ret_10",
    "volume_ratio",
    "volume_z",
    "range_position",
    "std_close_10",
    "liquidity_rank"
]

X = df[features]
y = df["target_buy_10"]

### Train/Test Split

In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False   # IMPORTANT for time-series data
)

### Check nulls or invalid values

In [88]:
import numpy as np

print("infinites :\n-----------\n", np.isinf(X_train).sum())
print("\nNans :\n------\n", np.isnan(X_train).sum())
print("\nis finite all ? \n--------------\n", np.isfinite(X_train).all())  # should return True for all columns

infinites :
-----------
 ret_5             0
ret_10            0
volume_ratio      0
volume_z          0
range_position    0
std_close_10      0
liquidity_rank    0
dtype: int64

Nans :
------
 ret_5             0
ret_10            0
volume_ratio      0
volume_z          0
range_position    0
std_close_10      0
liquidity_rank    0
dtype: int64

is finite all ? 
--------------
 ret_5             True
ret_10            True
volume_ratio      True
volume_z          True
range_position    True
std_close_10      True
liquidity_rank    True
dtype: bool


In [89]:
print(y_train.value_counts())

target_buy_10
0    6448
1    3261
Name: count, dtype: int64


### Train Model

In [90]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators = 200,     # Number of trees
    max_depth = None,          # Maximum depth of each tree
    random_state = 42,      # reproducible results
    n_jobs = -1,            # use all CPU cores (faster)
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight="balanced" # # Adjust weights inversely proportional to class frequencies (helps with imbalanced datasets)
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

### Evaluate Model
- **Accuracy**
   - overall correctness of model
   - Accuracy = Correct Predictions / Total predictions
- **Precision**
   - When model predicts this class, how often is it correct ?
   - Precision = True Positives / Predicted positives
   - Class 1 precision = 0.X --> Only X% of predicted BUY signals were actually correct
- **Recall**
  - Out of all real items of this class, how many did model find ?
  - Recall = True Positives / Actual Positives
  - Class 1 recall = 0.X --> Model detected X% of real BUY opportunities
- **F1-Score**
   - Balance between precision and recall
   - F1 = 2 × ( Precision × Recall ) / ( Precision + Recall )
   - Class 1 f1 = 0.X --> Overall prediction quality for BUY class is moderate
- **Support**
   - support = number of true samples
- **macro avg**
   - Macro average = simple average of classes
   - Treats both classes equally
   - Eg : Macro Precision = ( Precision0 + Precision1 ) / 2
   - Used when, you want fairness across classes
- **weighted avg**
   - Weighted average = average weighted by class size
   - Weighted = ( metric0 x n0 + metric1 × n1 ) / total
   - Used when, dataset is imbalanced and you want realistic performance

#### Which metric matters most?

| Goal | Best metric |
|------|-------------|
| Avoid false trades | precision |
| Find all good trades | recall |
| Balanced system | f1-score |
| Overall correctness | accuracy |

In [91]:
from sklearn.metrics import accuracy_score, classification_report

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.6725700164744646
              precision    recall  f1-score   support

           0       0.69      0.88      0.78      1564
           1       0.58      0.30      0.40       864

    accuracy                           0.67      2428
   macro avg       0.64      0.59      0.59      2428
weighted avg       0.65      0.67      0.64      2428



#### Save Model

In [ ]:
import joblib
import os

os.makedirs("pkls", exist_ok=True)
joblib.dump(model, "pkls/stock_model.pkl")

## Phase 4.5 (Tuning)
- We must optimize : scoring = "f1", actualy not accuracy

### 🌲 RandomForestClassifier (Model)
- A machine learning algorithm that makes predictions using many decision trees
- Instead of asking one expert, it asks many experts and takes a vote
- Single decision trees can be wrong, but Many trees together → more accurate and stable
- We don't use all varible for each tree we use sqrt of no of varible to train a particular tree
- then we get the majority vote

### 🔍 GridSearchCV (Hyperparameter tuner)
- A tool that automatically finds the best model settings
- in simple words - automatic parameter testerYou
- It  try all combinations and pick best

### 🔁 StratifiedKFold (Data splitting method)
- A smarter way to split data for validation
- Fold = data group we split into 80, 20 %, but if we have only 0 in a choosen group it dont know about 1, it split 0 group ino agian train test split no meaning
- Problem it solves
  - If your dataset is imbalanced 80% class 0, 20% class 1 --> Random splitting might produce : Fold 1 → 95% class 0, Fold 2 → 100% class 0
  - StratifiedKFold keeps same class ratio in every split > 80% class 0, 20% class 1

In [ ]:
# 108 combinations
# With CV=5 →
# 108 × 5 = 540 model trainings

n_estimators:      [100, 300, 500]
max_depth:         [5, 10, 20, None]
min_samples_split: [2, 5, 10]
min_samples_leaf:  [1, 2, 4]

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [100, 300, 500],   # Number of trees in the forest to test -> More trees = usually better but slower
    "max_depth": [5, 10, 20, None],    # Maximum depth of each tree: small depth → simple model / large depth → complex model / None → grow fully until pure
    "min_samples_split": [2, 5, 10],   # Minimum samples needed to split a node: smaller → more splits → more complex model / larger → fewer splits → simpler model
    "min_samples_leaf": [1, 2, 4]      # Minimum samples required in each leaf node: prevents tiny leaves / helps reduce overfitting
}

# random_state=42 → reproducible results
# n_jobs=-1 → use all CPU cores (faster)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Creates cross-validation splitter
# n_splits=5 → 5-fold validation
# Stratified → keeps class balance in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Creates a Grid Search object that will test all parameter combinations
grid = GridSearchCV(
    estimator=rf,   # Model to tune → RandomForest
    param_grid=param_grid,   # Search space → the combinations defined earlier
    scoring="f1",   # Metric used to judge models → F1 score >>> balances precision + recall, better than accuracy for imbalanced data
    cv=cv,   # Use custom cross-validation splitter
    verbose=3,   # Print progress while training, 1 - no, 2, more, 3- morea detailos than 2, .. so on..
    n_jobs=-1   # Run parameter search using all CPU cores
)

# Starts training
# For each hyperparameter combination : 
#    Train model on 4 folds
#    Test on remaining fold
#    Repeat 5 times
#    Compute average F1 score
grid.fit(X_train, y_train)

# Prints the best hyperparameters found.
print("Best Params:", grid.best_params_)

# Prints best average cross-validation F1 score
print("Best Score:", grid.best_score_)

# Stores the best trained model so you can use it
best_model = grid.best_estimator_

In [93]:
y_pred = best_model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.95      0.91      1564
           1       0.90      0.74      0.81       864

    accuracy                           0.88      2428
   macro avg       0.88      0.85      0.86      2428
weighted avg       0.88      0.88      0.88      2428

